# CAGE on Colab

Runs the v0.1 study on a Colab GPU (Runtime > Change runtime type > L4). The code is cloned to `/content/CAGE` on the runtime's own disk. Google Drive keeps only what must survive a disconnect, under `MyDrive/CAGE`: `datasets/` (the prepared Imagenette npz), `runs/` (checkpoints, manifests, logs) and `results/` (aggregates and figures).

Rerunning the training cell after a disconnect skips finished runs and restarts interrupted ones.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!test -d /content/CAGE || git clone -q https://github.com/KadirKess/CAGE.git /content/CAGE
!git -C /content/CAGE pull -q --ff-only
%cd /content/CAGE
!pip install -q uv && uv sync --locked -q
!git describe --always --dirty

In [ ]:
DRIVE = "/content/drive/MyDrive/CAGE"
CONFIG = "experiments/configs/v0.1.json"
STUDY = "v0.1"
JOBS = 1  # one job already saturates an L4
PYTHON = ".venv/bin/python"
RUNS = f"{DRIVE}/runs"
RESULTS = f"{DRIVE}/results"

In [ ]:
import os

dataset = "data/imagenette_128.npz"
cached = f"{DRIVE}/datasets/imagenette_128.npz"
os.makedirs(f"{DRIVE}/datasets", exist_ok=True)

if os.path.exists(dataset):
    print("Dataset already in place")
elif os.path.exists(cached):
    !mkdir -p data && cp {cached} {dataset}
else:
    !{PYTHON} scripts/prepare_imagenette.py --output {dataset}
    !cp {dataset} {cached}

Optional: time the training step, the loader and concurrent jobs on this GPU. Only needed on hardware other than the L4 the config was tuned on.

In [ ]:
# !{PYTHON} scripts/colab_benchmark.py

## Train

The dry run lists the 63 jobs without touching any data. Baselines train first; each denoiser waits for the WeakCNN of its seed.

In [ ]:
!{PYTHON} -m cage.run train --config {CONFIG} --output-root {RUNS} --dry-run

In [ ]:
!{PYTHON} -u -m cage.run train --config {CONFIG} --output-root {RUNS} --jobs {JOBS}

## Evaluate and aggregate

Paired test evaluation in strict FP32, then seed-level aggregates and the release figures.

In [ ]:
!{PYTHON} -u -m cage.run evaluate --study {STUDY} --output-root {RUNS}
!{PYTHON} -u -m cage.run aggregate --study {STUDY} --output-root {RUNS} --results-root {RESULTS}
!{PYTHON} -u -m cage.run analyze --study {STUDY} --output-root {RUNS} --results-root {RESULTS}

## Plots

These cells read the files on Drive with plain Python, so they also work on a fresh runtime without retraining.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

runs = Path(RUNS) / STUDY / "runs"
done = sorted(runs.glob("*/seed-*/manifest.json"))
print(f"{len(done)} / 63 runs complete")

# Training curves for one configuration, one line per seed
config_id = "mse_main"
fig, (loss_ax, acc_ax) = plt.subplots(1, 2, figsize=(12, 4))
for log in sorted(runs.glob(f"{config_id}/seed-*/metrics/*.json")):
    rows = json.loads(log.read_text())
    epochs = [row["epoch"] for row in rows]
    seed = log.parent.parent.name
    loss_ax.plot(epochs, [row["val_loss"] for row in rows], label=f"{seed} val")
    loss_ax.plot(epochs, [row["train_loss"] for row in rows], "--", alpha=0.5, label=f"{seed} train")
    if "cnn_accuracy_per_sigma" in rows[0]:
        for sigma in rows[0]["cnn_accuracy_per_sigma"]:
            acc_ax.plot(epochs, [row["cnn_accuracy_per_sigma"][sigma] for row in rows], label=f"{seed} sigma={sigma}")
loss_ax.set(title=f"{config_id} loss", xlabel="epoch")
acc_ax.set(title=f"{config_id} critic accuracy on denoised val", xlabel="epoch")
for ax in (loss_ax, acc_ax):
    ax.legend(fontsize=7)
plt.show()

In [ ]:
from IPython.display import Image, display

summary_path = Path(RESULTS) / STUDY / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("Paired accuracy difference vs reconstruction control at test sigma 0.3")
    for row in summary["paired_deltas_vs_control"]:
        if row["metric"] == "accuracy" and row["test_sigma"] == 0.3:
            ci = "n/a" if row["ci95"] is None else f"+/-{row['ci95']:.4f}"
            print(f"  {row['condition']}: {row['mean']:+.4f} {ci} (n={row['n']})")
else:
    print("No aggregate yet; run the evaluation cell first")

for figure in sorted((Path(RESULTS) / STUDY / "figures").glob("*.png")):
    print(figure.name)
    display(Image(filename=str(figure)))